In [1]:
a = 2

In [2]:
!pip install esp-ppq

In [5]:
import torch
import torch.nn as nn
from esp_ppq import espdl_quantize_onnx


    ___________ ____        ____  ____  ____
   / ____/ ___// __ \      / __ \/ __ \/ __ \
  / __/  \__ \/ /_/ /_____/ /_/ / /_/ / / / /
 / /___ ___/ / ____/_____/ ____/ ____/ /_/ /
/_____//____/_/         /_/   /_/    \___\_\




ImportError: cannot import name 'espdl_quantize_onnx' from 'esp_ppq' (c:\Users\shrib\anaconda3\envs\esp_env\Lib\site-packages\esp_ppq\__init__.py)

In [21]:
from esp_ppq.api import espdl_quantize_onnx

In [7]:
# 1. Define your model architecture (must match your training)
class GestureModel(nn.Module):
    def __init__(self):
        super(GestureModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(39, 20), nn.ReLU(),
            nn.Linear(20, 10), nn.ReLU(),
            nn.Linear(10, 3)
        )
    def forward(self, x): return self.net(x)

In [8]:
model = GestureModel()

In [10]:
tflite_model_path = "ei-esp32_gesture_detector-classifier-tensorflow-lite-float32-model.3.lite"

In [11]:
calibration_dataset = [torch.randn(1, 39) for _ in range(128)]

In [23]:
from esp_ppq.api import espdl_quantize_onnx



In [24]:
# 1. Define your model's input shape 
# Change this to match your Edge Impulse model's input!
# Format is [Batch, Channels, Height, Width] or [Batch, Features]
MY_INPUT_SHAPE = [1, 3, 224, 224]

In [25]:
dummy_calib_data = [torch.rand(MY_INPUT_SHAPE) for _ in range(8)]

In [27]:
import onnx

# Load your model
model = onnx.load("C:/Study/ESP32C/espdl/model.onnx")

# Print the input name and shape
for input in model.graph.input:
    print(f"Input Name: {input.name}")
    # Extract dimensions
    shape = [dim.dim_value for dim in input.type.tensor_type.shape.dim]
    print(f"Required Input Shape: {shape}")

Input Name: serving_default_x:0
Required Input Shape: [1, 39]


In [28]:
import torch
from esp_ppq.api import espdl_quantize_onnx

# 1. Update this to the shape you found in Step 1!
# If the output above was [1, 39], use [1, 39]
MY_INPUT_SHAPE = [1, 39] 

# 2. Re-create dummy data with the correct size
dummy_calib_data = [torch.rand(MY_INPUT_SHAPE) for _ in range(8)]

# 3. Run quantization
quant_ppq_graph = espdl_quantize_onnx(
    onnx_import_file="C:/Study/ESP32C/espdl/model.onnx",
    espdl_export_file="C:/Study/ESP32C/espdl/model.espdl",
    calib_dataloader=dummy_calib_data,
    calib_steps=8,
    input_shape=MY_INPUT_SHAPE,
    target='esp32s3',
    device='cpu'
)

[01:42:27] ConvTranspose Decomposition Pass Running ... Finished.
[01:42:27] PPQ Quantization Fusion Pass Running ...       Finished.
[01:42:27] PPQ Quantize Simplify Pass Running ...         Finished.
[01:42:27] PPQ Parameter Quantization Pass Running ...    Finished.
[01:42:27] PPQ Runtime Calibration Pass Running ...       

Calibration Progress(Phase 2): 100%|██████████| 8/8 [00:00<00:00, 311.57it/s]


Finished.
[01:42:27] PPQ Quantization Alignment Pass Running ...    Finished.
[01:42:27] PPQ Passive Parameter Quantization Running ... Finished.
--------- Network Snapshot ---------
Num of Op:                    [6]
Num of Quantized Op:          [6]
Num of Variable:              [13]
Num of Quantized Var:         [13]
------- Quantization Snapshot ------
Num of Quant Config:          [18]
ACTIVATED:                    [7]
OVERLAPPED:                   [7]
PASSIVE:                      [3]
FP32:                         [1]
Network Quantization Finished.


Analysing Graphwise Quantization Error(Phrase 1):: 100%|██████████| 8/8 [00:00<00:00, 285.97it/s]
Analysing Graphwise Quantization Error(Phrase 2):: 100%|██████████| 8/8 [00:00<00:00, 212.95it/s]


Layer                                                                                  | NOISE:SIGNAL POWER RATIO 
sequential/y_pred/MatMul;sequential/y_pred/BiasAdd_Gemm__15:                           | ████████████████████ | 0.025%
sequential/dense_1/MatMul;sequential/dense_1/Relu;sequential/dense_1/BiasAdd_Gemm__14: | ████                 | 0.013%
sequential/dense/MatMul;sequential/dense/Relu;sequential/dense/BiasAdd_Gemm__13:       |                      | 0.010%


Analysing Layerwise quantization error:: 100%|██████████| 3/3 [00:00<00:00, 190.08it/s]

Layer                                                                                  | NOISE:SIGNAL POWER RATIO 
sequential/dense/MatMul;sequential/dense/Relu;sequential/dense/BiasAdd_Gemm__13:       | ████████████████████ | 0.004%
sequential/y_pred/MatMul;sequential/y_pred/BiasAdd_Gemm__15:                           | ████████████         | 0.003%
sequential/dense_1/MatMul;sequential/dense_1/Relu;sequential/dense_1/BiasAdd_Gemm__14: |                      | 0.001%
[INFO][ESPDL][2026-04-18 01:42:27]:  Skip StatefulPartitionedCall:0 because it's not exportable
[INFO][ESPDL][2026-04-18 01:42:27]:  Skip StatefulPartitionedCall:0 because it's not exportable


[INFO][ESPDL][2026-04-18 01:42:27]:  Skip StatefulPartitionedCall:0 because it's not exportable
[INFO][ESPDL][2026-04-18 01:42:27]:  Skip StatefulPartitionedCall:0 because it's not exportable
